## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats
from sklearn.preprocessing import LabelEncoder

## Step 2: Load the placement dataset into a Pandas Dataframe.

In [14]:

df=pd.read_csv("data.csv")
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


In [16]:
df.shape

(148, 10)

In [17]:
df.head

<bound method NDFrame.head of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
7        8      M  64.00   Science     66.00   Sci&Tech     67.0   
..     ...    ...    ...       ...       ...        ...      ...   
209    210      M  72.00  Commerce     65.00  Comm&Mgmt     67.0   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
4

In [18]:
df.tail

<bound method NDFrame.tail of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
7        8      M  64.00   Science     66.00   Sci&Tech     67.0   
..     ...    ...    ...       ...       ...        ...      ...   
209    210      M  72.00  Commerce     65.00  Comm&Mgmt     67.0   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
4

In [19]:
df.sample(5)

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
89,90,F,75.00,Science,69.00,Sci&Tech,62.0,Mkt&HR,62.36,210000.0
39,40,M,68.00,Science,64.00,Sci&Tech,93.0,Mkt&Fin,62.56,411000.0
11,12,M,68.40,Commerce,78.30,Comm&Mgmt,60.0,Mkt&Fin,63.70,250000.0
1,2,M,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0
15,16,F,75.00,Commerce,69.00,Comm&Mgmt,72.0,Mkt&Fin,64.66,200000.0


In [20]:
df.describe()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary
count,148.000000,146.000000,147.000000,146.000000,148.000000,148.000000
mean,106.878378,70.000890,68.731973,73.140890,62.579392,288655.405405
std,60.682502,9.362426,6.539535,13.795521,5.884583,93457.452420
min,1.000000,50.830000,56.000000,50.000000,52.380000,200000.000000
25%,57.750000,63.000000,65.000000,60.000000,57.772500,240000.000000
50%,108.500000,68.200000,68.000000,72.000000,62.245000,265000.000000
75%,153.250000,75.750000,72.615000,85.000000,66.760000,300000.000000
max,214.000000,97.700000,91.000000,98.000000,77.890000,940000.000000


In [21]:
df.loc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [22]:
df.iloc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [23]:
df[0:2]

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,91.00,Commerce,58.00,Sci&Tech,55.0,Mkt&HR,58.80,270000.0
1,2,M,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0


In [24]:
df["degree_p"]

0      58.00
1      77.48
2      64.00
4      73.30
7      66.00
       ...  
209    65.00
210    77.60
211    72.00
212    73.00
213    58.00
Name: degree_p, Length: 148, dtype: float64

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [25]:
df.dropna(subset=["salary"],inplace=True)
df.isnull().sum()


sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

In [26]:
 df.shape 


(148, 10)

In [27]:
 df.head 


<bound method NDFrame.head of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
7        8      M  64.00   Science     66.00   Sci&Tech     67.0   
..     ...    ...    ...       ...       ...        ...      ...   
209    210      M  72.00  Commerce     65.00  Comm&Mgmt     67.0   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
4

In [28]:
 df.tail


<bound method NDFrame.tail of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
7        8      M  64.00   Science     66.00   Sci&Tech     67.0   
..     ...    ...    ...       ...       ...        ...      ...   
209    210      M  72.00  Commerce     65.00  Comm&Mgmt     67.0   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
4

In [29]:
 df.sample(5)


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
162,163,M,87.60,Commerce,77.25,Comm&Mgmt,75.20,Mkt&Fin,66.06,285000.0
85,86,F,89.83,Commerce,77.20,Comm&Mgmt,78.74,Mkt&Fin,76.18,400000.0
90,91,F,90.00,Commerce,82.00,Comm&Mgmt,92.00,Mkt&Fin,68.03,300000.0
187,188,M,65.50,Science,67.00,Sci&Tech,95.00,Mkt&Fin,64.86,280000.0
74,75,M,64.80,Commerce,70.20,Comm&Mgmt,84.27,Mkt&Fin,67.20,336000.0


In [30]:
 df.describe()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary
count,148.000000,146.000000,147.000000,146.000000,148.000000,148.000000
mean,106.878378,70.000890,68.731973,73.140890,62.579392,288655.405405
std,60.682502,9.362426,6.539535,13.795521,5.884583,93457.452420
min,1.000000,50.830000,56.000000,50.000000,52.380000,200000.000000
25%,57.750000,63.000000,65.000000,60.000000,57.772500,240000.000000
50%,108.500000,68.200000,68.000000,72.000000,62.245000,265000.000000
75%,153.250000,75.750000,72.615000,85.000000,66.760000,300000.000000
max,214.000000,97.700000,91.000000,98.000000,77.890000,940000.000000


#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [31]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [32]:
df.dropna(subset = ["salary"],inplace=True)
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [33]:
df["hsc_p"] = df["hsc_p"].fillna(df["hsc_p"].mean())
df["degree_p"] = df["degree_p"].fillna(df["degree_p"].mean())
df["etest_p"] = df["etest_p"].fillna(df["etest_p"].mean())
df.isnull().sum()

sl_no             0
gender            0
hsc_p             0
hsc_s             0
degree_p          0
degree_t          0
etest_p           0
specialisation    0
mba_p             0
salary            0
dtype: int64

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [34]:
c=["hsc_p","degree_p","etest_p","salary"]
s1 = StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.533482e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [35]:
s2 = MinMaxScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.341443e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [36]:
colmn_to_check = ["salary"]
z_score=stats.zscore(df[colmn_to_check])
# display(z_score)
u=(z_score>3)
l=(z_score<-3)
index=u|l
clean_df = df[~index]
df.info()
clean_df.info()
# print(u)
# print(l)
# print(index)


<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.8 KB
<class 'pandas.DataFrame'>
Index: 145 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           145 non-null    int64  
 1   gender          145 non-null    str    
 2   hsc_p           145 non-null    float6

### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [37]:

l1=df["salary"].quantile(0.05)
u1=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"]=df_capped["salary"].clip(l1,u1)
df_capped.info()
df.head()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.8 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.341443e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




In [38]:
from sklearn.preprocessing import LabelEncoder

L1 = LabelEncoder()

df["gender"] = L1.fit_transform(df["gender"])
df["hsc_s"] = L1.fit_transform(df["hsc_s"])
df["degree_t"] = L1.fit_transform(df["degree_t"])
df["specialisation"] = L1.fit_transform(df["specialisation"])

df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,1,2.265997e+00,1,-1.652293,2,-1.328518,1,58.80,-0.200292
1,2,1,8.987875e-01,2,1.346845,2,0.978332,0,66.28,-0.951839
2,3,1,1.341443e-15,0,-0.728534,0,0.136149,0,57.80,-0.415019
4,5,1,3.883770e-01,1,0.703293,0,1.732635,0,55.50,1.463849
7,8,1,-6.475513e-01,2,-0.420614,2,-0.449718,0,62.14,-0.393547


## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [39]:


from sklearn.preprocessing import OneHotEncoder

L1 = OneHotEncoder(sparse_output=False, dtype=int)

encoded = L1.fit_transform(df[["gender", "hsc_s", "degree_t", "specialisation"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=L1.get_feature_names_out(["gender", "hsc_s", "degree_t", "specialisation"]),
    index=df.index
)

df = pd.concat([df.drop(columns=["gender", "hsc_s", "degree_t", "specialisation"]), encoded_df], axis=1)

df.head()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary,gender_0,gender_1,hsc_s_0,hsc_s_1,hsc_s_2,degree_t_0,degree_t_1,degree_t_2,specialisation_0,specialisation_1
0,1,2.265997e+00,-1.652293,-1.328518,58.80,-0.200292,0,1,0,1,0,0,0,1,0,1
1,2,8.987875e-01,1.346845,0.978332,66.28,-0.951839,0,1,0,0,1,0,0,1,1,0
2,3,1.341443e-15,-0.728534,0.136149,57.80,-0.415019,0,1,1,0,0,1,0,0,1,0
4,5,3.883770e-01,0.703293,1.732635,55.50,1.463849,0,1,0,1,0,1,0,0,1,0
7,8,-6.475513e-01,-0.420614,-0.449718,62.14,-0.393547,0,1,0,0,1,0,0,1,1,0


# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

In [5]:
# STEP 1: LOAD AND EXPLORE THE DATASET

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("automobile.csv")

# Display first 10 rows
print("First 10 rows:")
display(df.head(10))

# Display shape
print("Shape of dataset:", df.shape)

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Display data types
print("\nData types:")
print(df.dtypes)

# Descriptive statistics
print("\nDescriptive statistics:")
display(df.describe(include="all"))

# Identify numerical and categorical columns
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("\nNumerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

# Display unique values in categorical columns
print("\nUnique values in categorical columns:")
for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].unique())

First 10 rows:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa


Shape of dataset: (398, 9)

Column names:
['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']

Data types:
name                str
mpg             float64
cylinders       float64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin              str
dtype: object

Descriptive statistics:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000,398
unique,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,ford pinto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,usa
freq,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,249
mean,NaN,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050,NaN
std,NaN,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627,NaN
min,NaN,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,NaN
25%,NaN,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000,NaN
50%,NaN,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000,NaN
75%,NaN,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000,NaN



Numerical columns:
['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']

Categorical columns:
['name', 'origin']

Unique values in categorical columns:

name:
<StringArray>
[ 'chevrolet chevelle malibu',          'buick skylark 320',
         'plymouth satellite',              'amc rebel sst',
                'ford torino',           'ford galaxie 500',
           'chevrolet impala',          'plymouth fury iii',
           'pontiac catalina',         'amc ambassador dpl',
 ...
 'chrysler lebaron medallion',             'ford granada l',
           'toyota celica gt',          'dodge charger 2.2',
           'chevrolet camaro',            'ford mustang gl',
                  'vw pickup',              'dodge rampage',
                'ford ranger',                 'chevy s-10']
Length: 305, dtype: str

origin:
<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str


In [6]:
# STEP 2: CHECK MISSING VALUES

# Number of missing values
missing_count = df.isnull().sum()

# Percentage of missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_count,
    "Percentage": missing_percentage
})

print("Missing values summary:")
display(missing_summary)

Missing values summary:


,Missing Values,Percentage
name,0,0.000000
mpg,0,0.000000
cylinders,3,0.753769
displacement,3,0.753769
horsepower,12,3.015075
weight,2,0.502513
acceleration,3,0.753769
model_year,0,0.000000
origin,0,0.000000


In [7]:
# STEP 3: HANDLE MISSING VALUES

# Replace missing numerical values with median
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

# Replace missing categorical values with mode
for col in categorical_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Verify missing values
print("Missing values after handling:")
display(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values after handling:


name            0
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64


Total missing values: 0


In [8]:
# STEP 4: REMOVE DUPLICATE RECORDS

# Check duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

# Remove duplicates
df = df.drop_duplicates()

# Verify
print("Shape after removing duplicates:", df.shape)
print("Duplicate records after removal:", df.duplicated().sum())

Number of duplicate records: 0
Shape after removing duplicates: (398, 9)
Duplicate records after removal: 0


In [9]:
# STEP 5: CLEAN THE HORSEPOWER COLUMN

# Display non-numeric values
print("Unique values in horsepower before cleaning:")
print(df["horsepower"].unique())

# Replace '?' with NaN
df["horsepower"] = df["horsepower"].replace("?", np.nan)

# Convert horsepower to numeric
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")

# Fill missing horsepower values with median
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Verify
print("\nHorsepower data type:")
print(df["horsepower"].dtype)

print("\nMissing horsepower values:")
print(df["horsepower"].isnull().sum())

print("\nFirst 10 horsepower values:")
print(df["horsepower"].head(10))

Unique values in horsepower before cleaning:
[130.  165.  150.  140.  198.  220.  215.   92.5 190.  170.  160.  225.
  95.   97.   85.   88.   87.   90.  113.  200.  210.  193.  100.  105.
 175.  153.  180.  110.   72.   86.   70.   76.   65.   69.   60.   80.
  54.  208.  155.  112.   92.  145.  137.  158.   46.  167.   94.  107.
 230.   49.   75.   91.  122.   67.   83.   78.   52.   61.   93.  148.
 129.   96.   71.   98.  115.   53.   81.   79.  120.  152.  102.  108.
  68.   58.  149.   89.   63.   48.   66.  139.  103.  125.  133.  138.
 135.  142.   77.   62.  132.   84.   64.   74.  116.   82. ]

Horsepower data type:
float64

Missing horsepower values:
0

First 10 horsepower values:
0    130.0
1    165.0
2    150.0
3    150.0
4    140.0
5    198.0
6    220.0
7    215.0
8     92.5
9    190.0
Name: horsepower, dtype: float64


In [10]:
# STEP 6: TRANSFORM THE ORIGIN COLUMN

print("Unique origin values before transformation:")
print(df["origin"].unique())

# Convert numeric origin values into meaningful labels
origin_mapping = {
    1: "usa",
    2: "europe",
    3: "japan"
}

df["origin"] = df["origin"].map(origin_mapping)

print("\nUnique origin values after transformation:")
print(df["origin"].unique())

print("\nOrigin value counts:")
print(df["origin"].value_counts())

Unique origin values before transformation:
<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str

Unique origin values after transformation:
<StringArray>
[nan]
Length: 1, dtype: str

Origin value counts:
Series([], Name: count, dtype: int64)


In [11]:
# STEP 7: CREATE WEIGHT_KG

# Convert pounds to kilograms
df["weight_kg"] = df["weight"] * 0.453592

print("First 10 rows with weight conversion:")
display(df[["weight", "weight_kg"]].head(10))

First 10 rows with weight conversion:


,weight,weight_kg
0,3504.0,1589.386368
1,3693.0,1675.115256
2,3436.0,1558.542112
3,3433.0,1557.181336
4,3449.0,1564.438808
5,4341.0,1969.042872
6,2797.5,1268.923620
7,4312.0,1955.888704
8,4425.0,2007.144600
9,3850.0,1746.329200


In [12]:
# STEP 8: CREATE MPG_CATEGORY

def categorize_mpg(mpg):
    if mpg < 20:
        return "Low"
    elif mpg < 30:
        return "Medium"
    else:
        return "High"

df["mpg_category"] = df["mpg"].apply(categorize_mpg)

print("MPG categories:")
print(df["mpg_category"].value_counts())

print("\nFirst 10 rows:")
display(df[["mpg", "mpg_category"]].head(10))

MPG categories:
mpg_category
Medium    155
Low       151
High       92
Name: count, dtype: int64

First 10 rows:


,mpg,mpg_category
0,18.0,Low
1,15.0,Low
2,18.0,Low
3,16.0,Low
4,17.0,Low
5,15.0,Low
6,14.0,Low
7,14.0,Low
8,14.0,Low
9,15.0,Low


In [13]:
# STEP 9: CREATE VEHICLE_AGE

current_year = 2026

df["vehicle_age"] = current_year - df["model_year"]

print("First 10 rows:")
display(df[["model_year", "vehicle_age"]].head(10))

First 10 rows:


,model_year,vehicle_age
0,70,1956
1,70,1956
2,70,1956
3,70,1956
4,70,1956
5,70,1956
6,70,1956
7,70,1956
8,70,1956
9,70,1956


In [14]:
# STEP 10: RENAME COLUMNS

df = df.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year"
})

print("Updated column names:")
print(df.columns.tolist())

Updated column names:
['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg', 'mpg_category', 'vehicle_age']


In [15]:
# STEP 11: FILTER THE DATA

# Vehicles with MPG > 30
print("Vehicles with miles_per_gallon > 30:")
display(df[df["miles_per_gallon"] > 30])

# Vehicles with horsepower > 150
print("\nVehicles with hp > 150:")
display(df[df["hp"] > 150])

# Vehicles with cylinders >= 6
print("\nVehicles with cylinders >= 6:")
display(df[df["cylinders"] >= 6])

# Vehicles manufactured after 1980
print("\nVehicles manufactured after 1980:")
display(df[df["year"] > 1980])

# Vehicles originating from USA
print("\nVehicles originating from USA:")
display(df[df["origin"] == "usa"])

Vehicles with miles_per_gallon > 30:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
53,toyota corolla 1200,31.0,4.0,71.0,65.0,1773.0,19.0,71,NaN,804.218616,High,1955
54,datsun 1200,35.0,4.0,72.0,69.0,1613.0,18.0,71,NaN,731.643896,High,1955
129,datsun b210,31.0,4.0,79.0,67.0,1950.0,19.0,74,NaN,884.504400,High,1952
131,toyota corolla 1200,32.0,4.0,71.0,92.5,1836.0,15.5,74,NaN,832.794912,High,1952
144,toyota corona,31.0,4.0,76.0,52.0,1649.0,16.5,74,NaN,747.973208,High,1952
...,...,...,...,...,...,...,...,...,...,...,...,...
390,toyota celica gt,32.0,4.0,144.0,96.0,2665.0,13.9,82,NaN,1208.822680,High,1944
391,dodge charger 2.2,36.0,4.0,135.0,84.0,2370.0,13.0,82,NaN,1075.013040,High,1944
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82,NaN,966.150960,High,1944
395,dodge rampage,32.0,4.0,135.0,84.0,2295.0,11.6,82,NaN,1040.993640,High,1944



Vehicles with hp > 150:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,NaN,1969.042872,Low,1956
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,NaN,1268.923620,Low,1956
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,NaN,1955.888704,Low,1956
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,NaN,1746.329200,Low,1956
10,dodge challenger se,15.0,8.0,383.0,170.0,3563.0,10.0,70,NaN,1616.148296,Low,1956
11,plymouth 'cuda 340,14.0,8.0,340.0,160.0,3609.0,8.0,70,NaN,1637.013528,Low,1956
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70,NaN,1399.784912,Low,1956
25,ford f250,10.0,8.0,360.0,215.0,4615.0,14.0,70,NaN,2093.327080,Low,1956
26,chevy c20,10.0,8.0,307.0,200.0,4376.0,15.0,70,NaN,1984.918592,Low,1956



Vehicles with cylinders >= 6:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,Low,1956
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,Low,1956
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,Low,1956
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,NaN,1969.042872,Low,1956
...,...,...,...,...,...,...,...,...,...,...,...,...
365,ford granada gl,20.2,6.0,200.0,88.0,3060.0,17.1,81,NaN,1387.991520,Medium,1945
366,chrysler lebaron salon,17.6,6.0,225.0,85.0,3465.0,16.6,81,NaN,1571.696280,Low,1945
386,buick century limited,25.0,6.0,181.0,110.0,2945.0,16.4,82,NaN,1335.828440,Medium,1944
387,oldsmobile cutlass ciera (diesel),38.0,6.0,262.0,85.0,3015.0,17.0,82,NaN,1367.579880,High,1944



Vehicles manufactured after 1980:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age



Vehicles originating from USA:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age


In [16]:
# STEP 12: LABEL ENCODING

from sklearn.preprocessing import LabelEncoder

# Create encoder
label_encoder = LabelEncoder()

# Apply Label Encoding
df["origin_encoded"] = label_encoder.fit_transform(df["origin"])

# Display original and encoded values
print("Original and encoded origin values:")
display(df[["origin", "origin_encoded"]].drop_duplicates().sort_values("origin_encoded"))

# Display category-to-label mapping
mapping = dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
))

print("\nCategory-to-label mapping:")
print(mapping)

Original and encoded origin values:


,origin,origin_encoded
0,NaN,0



Category-to-label mapping:
{nan: np.int64(0)}


In [17]:
# STEP 13: ONE-HOT ENCODING

# Apply one-hot encoding
origin_one_hot = pd.get_dummies(
    df["origin"],
    prefix="origin",
    dtype=int
)

print("One-Hot Encoded values:")
display(origin_one_hot.head(10))

# Combine with dataframe if required
df_one_hot = pd.concat([df, origin_one_hot], axis=1)

print("\nColumns after One-Hot Encoding:")
print(df_one_hot.columns.tolist())

print("\nComparison:")
print("Label Encoding creates one numerical column.")
print("One-Hot Encoding creates separate binary columns for each category.")
print("\nFor 'origin', One-Hot Encoding is more appropriate because")
print("USA, Europe and Japan are nominal categories and have no natural order.")

One-Hot Encoded values:


""
0
1
2
3
4
5
6
7
8
9



Columns after One-Hot Encoding:
['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg', 'mpg_category', 'vehicle_age', 'origin_encoded']

Comparison:
Label Encoding creates one numerical column.
One-Hot Encoding creates separate binary columns for each category.

For 'origin', One-Hot Encoding is more appropriate because
USA, Europe and Japan are nominal categories and have no natural order.


In [18]:
# STEP 14: OUTLIER DETECTION USING Z-SCORES

from scipy.stats import zscore

# Select numerical features
numeric_data = df.select_dtypes(include=np.number)

# Calculate Z-scores
z_scores = pd.DataFrame(
    zscore(numeric_data, nan_policy="omit"),
    columns=numeric_data.columns,
    index=df.index
)

# Identify outliers where absolute Z-score > 3
outlier_mask = z_scores.abs() > 3

# Count outliers in each numerical column
outlier_counts = outlier_mask.sum()

print("Number of outliers in each numerical column:")
display(outlier_counts.to_frame("Outlier Count"))

# Display rows containing at least one outlier
outlier_rows = df[outlier_mask.any(axis=1)]

print("\nRows containing outliers:")
display(outlier_rows)

# Decision
print("\nDecision:")
print("Outliers should generally be retained unless they are confirmed to")
print("be errors or impossible values. In automobile data, extreme values")
print("may represent genuine vehicles, so they should not automatically be removed.")

Number of outliers in each numerical column:


,Outlier Count
miles_per_gallon,0
cylinders,0
displacement,0
hp,4
weight_lbs,0
acceleration,2
year,0
weight_kg,0
vehicle_age,0
origin_encoded,0



Rows containing outliers:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age,origin_encoded
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,NaN,1268.923620,Low,1956,0
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70,NaN,1399.784912,Low,1956,0
95,buick electra 225 custom,12.0,8.0,455.0,225.0,4951.0,11.0,73,NaN,2245.733992,Low,1953,0
116,pontiac grand prix,16.0,8.0,400.0,230.0,4278.0,9.5,73,NaN,1940.466576,Low,1953,0
299,peugeot 504,27.2,4.0,141.0,71.0,3190.0,24.8,79,NaN,1446.958480,Medium,1947,0
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82,NaN,966.150960,High,1944,0



Decision:
Outliers should generally be retained unless they are confirmed to
be errors or impossible values. In automobile data, extreme values
may represent genuine vehicles, so they should not automatically be removed.


In [19]:
# STEP 15: STANDARDIZATION USING STANDARDSCALER

from sklearn.preprocessing import StandardScaler

# Select numerical features
numerical_features = df.select_dtypes(include=np.number).columns.tolist()

# Create scaler
standard_scaler = StandardScaler()

# Apply standardization
standardized_data = standard_scaler.fit_transform(df[numerical_features])

# Convert to DataFrame
df_standardized = pd.DataFrame(
    standardized_data,
    columns=numerical_features,
    index=df.index
)

print("Standardized values:")
display(df_standardized.head(10))

# Verify mean and standard deviation
print("\nMean of standardized features:")
display(df_standardized.mean())

print("\nStandard deviation of standardized features:")
display(df_standardized.std())

Standardized values:


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded
0,-0.706439,1.515897,1.096516,0.694154,0.641001,-1.301636,-1.627426,0.641001,1.627426,0.0
1,-1.090751,1.515897,1.510055,1.627150,0.865428,-1.484357,-1.627426,0.865428,1.627426,0.0
2,-0.706439,1.515897,1.202305,1.227295,0.560255,-1.667078,-1.627426,0.560255,1.627426,0.0
3,-0.962647,1.515897,1.067664,1.227295,0.556693,-1.301636,-1.627426,0.556693,1.627426,0.0
4,-0.834543,-0.847774,1.048430,0.960725,0.575692,-1.849799,-1.627426,0.575692,1.627426,0.0
5,-1.090751,1.515897,2.269812,2.506832,1.634890,-2.032521,-1.627426,1.634890,1.627426,0.0
6,-1.218855,1.515897,2.510242,3.093287,-0.197927,-2.397963,-1.627426,-0.197927,1.627426,0.0
7,-1.218855,1.515897,2.375601,2.960002,1.600455,-2.580684,-1.627426,1.600455,1.627426,0.0
8,-1.218855,1.515897,2.519859,-0.305484,1.734636,-2.032521,-1.627426,1.734636,1.627426,0.0
9,-1.090751,1.515897,1.894742,2.293576,1.051856,-2.580684,-1.627426,1.051856,1.627426,0.0



Mean of standardized features:


miles_per_gallon    7.141133e-17
cylinders          -1.963812e-16
displacement       -8.926416e-17
hp                  3.570567e-17
weight_lbs         -1.249698e-16
acceleration       -6.427020e-16
year               -1.642461e-15
weight_kg           1.963812e-16
vehicle_age        -2.520820e-14
origin_encoded      0.000000e+00
dtype: float64


Standard deviation of standardized features:


miles_per_gallon    1.001259
cylinders           1.001259
displacement        1.001259
hp                  1.001259
weight_lbs          1.001259
acceleration        1.001259
year                1.001259
weight_kg           1.001259
vehicle_age         1.001259
origin_encoded      0.000000
dtype: float64

In [20]:
# STEP 16: NORMALIZATION USING MINMAXSCALER

from sklearn.preprocessing import MinMaxScaler

# Create MinMaxScaler
minmax_scaler = MinMaxScaler()

# Apply normalization
normalized_data = minmax_scaler.fit_transform(df[numerical_features])

# Convert to DataFrame
df_normalized = pd.DataFrame(
    normalized_data,
    columns=numerical_features,
    index=df.index
)

print("Normalized values:")
display(df_normalized.head(10))

# Verify minimum and maximum
print("\nMinimum values:")
display(df_normalized.min())

print("\nMaximum values:")
display(df_normalized.max())

print("\nComparison:")
print("Standardization transforms data to approximately mean = 0 and SD = 1.")
print("Normalization transforms data to a fixed range, usually 0 to 1.")
print("Standardization is useful for algorithms sensitive to variance and scale.")
print("Normalization is useful when a fixed range is required, such as in some neural networks.")

Normalized values:


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded
0,0.239362,1.0,0.617571,0.456522,0.536150,0.238095,0.0,0.536150,1.0,0.0
1,0.159574,1.0,0.728682,0.646739,0.589736,0.208333,0.0,0.589736,1.0,0.0
2,0.239362,1.0,0.645995,0.565217,0.516870,0.178571,0.0,0.516870,1.0,0.0
3,0.186170,1.0,0.609819,0.565217,0.516019,0.238095,0.0,0.516019,1.0,0.0
4,0.212766,0.2,0.604651,0.510870,0.520556,0.148810,0.0,0.520556,1.0,0.0
5,0.159574,1.0,0.932817,0.826087,0.773462,0.119048,0.0,0.773462,1.0,0.0
6,0.132979,1.0,0.997416,0.945652,0.335838,0.059524,0.0,0.335838,1.0,0.0
7,0.132979,1.0,0.961240,0.918478,0.765240,0.029762,0.0,0.765240,1.0,0.0
8,0.132979,1.0,1.000000,0.252717,0.797278,0.119048,0.0,0.797278,1.0,0.0
9,0.159574,1.0,0.832041,0.782609,0.634250,0.029762,0.0,0.634250,1.0,0.0



Minimum values:


miles_per_gallon    0.0
cylinders           0.0
displacement        0.0
hp                  0.0
weight_lbs          0.0
acceleration        0.0
year                0.0
weight_kg           0.0
vehicle_age         0.0
origin_encoded      0.0
dtype: float64


Maximum values:


miles_per_gallon    1.0
cylinders           1.0
displacement        1.0
hp                  1.0
weight_lbs          1.0
acceleration        1.0
year                1.0
weight_kg           1.0
vehicle_age         1.0
origin_encoded      0.0
dtype: float64


Comparison:
Standardization transforms data to approximately mean = 0 and SD = 1.
Normalization transforms data to a fixed range, usually 0 to 1.
Standardization is useful for algorithms sensitive to variance and scale.
Normalization is useful when a fixed range is required, such as in some neural networks.


In [21]:
# STEP 17: CREATE X AND Y VARIABLES

# Select independent variables
X = df.drop(columns=["miles_per_gallon", "mpg_category"])

# Select target variable
Y = df["miles_per_gallon"]

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

print("\nFirst 5 rows of X:")
display(X.head())

print("\nFirst 5 values of Y:")
display(Y.head())

# Save X and Y into one CSV file
XY = X.copy()
XY["miles_per_gallon"] = Y

XY.to_csv("automobile_X_Y.csv", index=False)

print("\nautomobile_X_Y.csv saved successfully.")

# Load the CSV file again
XY_loaded = pd.read_csv("automobile_X_Y.csv")

print("\nFirst 5 rows after loading CSV:")
display(XY_loaded.head())

Shape of X: (398, 11)
Shape of Y: (398,)

First 5 rows of X:


,name,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,vehicle_age,origin_encoded
0,chevrolet chevelle malibu,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,1956,0
1,buick skylark 320,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,1956,0
2,plymouth satellite,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,1956,0
3,amc rebel sst,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,1956,0
4,ford torino,4.0,302.0,140.0,3449.0,10.5,70,NaN,1564.438808,1956,0



First 5 values of Y:


0    18.0
1    15.0
2    18.0
3    16.0
4    17.0
Name: miles_per_gallon, dtype: float64


automobile_X_Y.csv saved successfully.

First 5 rows after loading CSV:


,name,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,vehicle_age,origin_encoded,miles_per_gallon
0,chevrolet chevelle malibu,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,1956,0,18.0
1,buick skylark 320,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,1956,0,15.0
2,plymouth satellite,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,1956,0,18.0
3,amc rebel sst,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,1956,0,16.0
4,ford torino,4.0,302.0,140.0,3449.0,10.5,70,NaN,1564.438808,1956,0,17.0


In [22]:
# STEP 18: SAVE THE FINAL PREPROCESSED DATASET

# Combine processed features and target variable
final_df = df.copy()

# Display final dataset
print("Final preprocessed dataset:")
display(final_df.head(10))

print("\nShape of final dataset:")
print(final_df.shape)

# Check for missing values
print("\nMissing values in final dataset:")
display(final_df.isnull().sum())

print("\nTotal missing values:")
print(final_df.isnull().sum().sum())

# Save final dataset
final_df.to_csv("automobile_preprocessed.csv", index=False)

print("\nautomobile_preprocessed.csv saved successfully.")

Final preprocessed dataset:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age,origin_encoded
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,Low,1956,0
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956,0
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,Low,1956,0
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,Low,1956,0
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,NaN,1564.438808,Low,1956,0
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,NaN,1969.042872,Low,1956,0
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,NaN,1268.923620,Low,1956,0
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,NaN,1955.888704,Low,1956,0
8,pontiac catalina,14.0,8.0,455.0,92.5,4425.0,10.0,70,NaN,2007.144600,Low,1956,0
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,NaN,1746.329200,Low,1956,0



Shape of final dataset:
(398, 13)

Missing values in final dataset:


name                  0
miles_per_gallon      0
cylinders             0
displacement          0
hp                    0
weight_lbs            0
acceleration          0
year                  0
origin              398
weight_kg             0
mpg_category          0
vehicle_age           0
origin_encoded        0
dtype: int64


Total missing values:
398

automobile_preprocessed.csv saved successfully.


In [23]:
# STEP 19: FINAL VERIFICATION

print("=" * 60)
print("FINAL DATASET VERIFICATION")
print("=" * 60)

print("\nShape:")
print(final_df.shape)

print("\nColumns:")
print(final_df.columns.tolist())

print("\nData types:")
print(final_df.dtypes)

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nDuplicate rows:")
print(final_df.duplicated().sum())

print("\nFirst 5 rows:")
display(final_df.head())

print("\nFinal dataset saved as: automobile_preprocessed.csv")

FINAL DATASET VERIFICATION

Shape:
(398, 13)

Columns:
['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg', 'mpg_category', 'vehicle_age', 'origin_encoded']

Data types:
name                    str
miles_per_gallon    float64
cylinders           float64
displacement        float64
hp                  float64
weight_lbs          float64
acceleration        float64
year                  int64
origin                  str
weight_kg           float64
mpg_category            str
vehicle_age           int64
origin_encoded        int64
dtype: object

Missing values:
name                  0
miles_per_gallon      0
cylinders             0
displacement          0
hp                    0
weight_lbs            0
acceleration          0
year                  0
origin              398
weight_kg             0
mpg_category          0
vehicle_age           0
origin_encoded        0
dtype: int64

Duplicate rows:
0

First 5 rows:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age,origin_encoded
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,Low,1956,0
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956,0
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,Low,1956,0
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,Low,1956,0
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,NaN,1564.438808,Low,1956,0



Final dataset saved as: automobile_preprocessed.csv
